# FineVideo × YOLO11l — Kaggle runner
Settings needed: **Accelerator = GPU T4 x2**, **Internet = On**, and a Kaggle Secret named **`HF_TOKEN`**
(Add-ons → Secrets) holding your Hugging Face read token.

1. First set `MODE = "test"` and run all cells interactively (about 20–30 min). Check that it PASSes and note the time per file.
2. Then set `MODE = "full"` and click **Save Version → Save & Run All (Commit)**. Close the browser and come back later.
3. Download `results.jsonl.gz` and `run_info.txt` from the notebook's **Output** tab.

In [ ]:
# ===== SETTINGS =====
MODE = "test"        # "test" first, then "full"
REPO_URL = "https://github.com/YOUR_USERNAME/YOUR_REPO.git"   # <-- your public repo
SAMPLE_FPS = 5       # frames per second sent to YOLO (5 is safe; 3 is faster)
BATCH = 32
TEST_VIDEOS = 6      # videos used for the every-frame correctness check in test mode

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!nproc; free -g | head -2; df -h /tmp /kaggle/working | cat
!lscpu | grep "Model name"

In [ ]:
!pip install -q ultralytics==8.3.253 av==16.0.1 pyarrow==20.0.0 huggingface_hub
import torch, ultralytics, av
print(torch.__version__, torch.cuda.device_count(), "GPUs", ultralytics.__version__, av.__version__)

In [ ]:
import os
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
print("HF token loaded:", len(os.environ["HF_TOKEN"]) > 0)

In [ ]:
import os
os.chdir("/kaggle/working")
if not os.path.exists("code"):
    !git clone -q {REPO_URL} code
os.chdir("/kaggle/working/code")
!python -c "from ultralytics import YOLO; YOLO('yolo11l.pt')"
!ls -la

In [ ]:
import subprocess, time, sys, os, torch
NGPU = max(1, torch.cuda.device_count())

def launch(cmd, gpu, log):
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu))
    return subprocess.Popen(cmd, env=env, stdout=open(log, "w"), stderr=subprocess.STDOUT)

def wait_all(procs, logs, every=300):
    while any(p.poll() is None for p in procs):
        time.sleep(every)
        for l in logs:
            print(f"--- {l}"); print("".join(open(l).readlines()[-3:]), flush=True)
    for l in logs:
        print(f"=== {l}"); print("".join(open(l).readlines()[-5:]))

In [ ]:
# ===== TEST MODE: correctness check + timing for one file =====
if MODE == "test":
    from huggingface_hub import hf_hub_download
    t = time.time()
    hf_hub_download("HuggingFaceFV/finevideo", "data/train-00000-of-01357.parquet",
                    repo_type="dataset", local_dir="/tmp/test0", token=os.environ["HF_TOKEN"])
    print(f"download of one file: {time.time()-t:.0f} s")
    # GPU 0: every-frame reference on the first TEST_VIDEOS videos
    p1 = launch([sys.executable, "run.py", "--input", "/tmp/test0/data", "--output", "/kaggle/working/ref_test.jsonl",
                 "--sample-fps", "0", "--limit", str(TEST_VIDEOS), "--batch", str(BATCH)], 0, "/kaggle/working/log_ref.txt")
    # GPU 1 (or 0): the fast version on the WHOLE first file, timed
    t0 = time.time()
    p2 = launch([sys.executable, "run.py", "--input", "/tmp/test0/data", "--output", "/kaggle/working/fast_file0.jsonl",
                 "--sample-fps", str(SAMPLE_FPS), "--batch", str(BATCH)], 1 % NGPU, "/kaggle/working/log_fast.txt")
    p2.wait(); t_file = time.time() - t0
    print(f"\nONE FILE took {t_file/60:.1f} min with SAMPLE_FPS={SAMPLE_FPS}")
    print(f"estimated full run on {NGPU} GPU(s): {t_file*50/NGPU/3600:.1f} hours")
    wait_all([p1], ["/kaggle/working/log_ref.txt"], every=60)
    !python check.py /kaggle/working/ref_test.jsonl /kaggle/working/fast_file0.jsonl

In [ ]:
# ===== FULL MODE: all 50 files, split across the GPUs =====
if MODE == "full":
    T0 = time.time()
    procs, logs = [], []
    for g in range(NGPU):
        log = f"/kaggle/working/log_gpu{g}.txt"
        procs.append(launch([sys.executable, "kaggle_job.py", "--gpu", str(g), "--shard", str(g),
                             "--nshards", str(NGPU), "--out", "/kaggle/working/results_parts",
                             "--sample-fps", str(SAMPLE_FPS), "--batch", str(BATCH)], g, log))
        logs.append(log)
    wait_all(procs, logs)
    TOTAL = time.time() - T0
    print(f"FULL RUN took {TOTAL/3600:.2f} hours")

In [ ]:
# ===== MERGE + SUMMARY =====
if MODE == "full":
    import glob, gzip, json
    parts = sorted(glob.glob("/kaggle/working/results_parts/*.jsonl"))
    done = glob.glob("/kaggle/working/results_parts/*.done")
    videos, errors, lines = set(), 0, 0
    with gzip.open("/kaggle/working/results.jsonl.gz", "wt") as out:
        for p in parts:
            for line in open(p):
                d = json.loads(line); videos.add(d["video"]); errors += "error" in d; lines += 1
                out.write(line)
    info = (f"files finished: {len(done)}/50\nvideos: {len(videos)}\nerror lines: {errors}\n"
            f"result lines: {lines}\nsample_fps: {SAMPLE_FPS}\nGPUs used: {NGPU}\n"
            f"full run wall time: {TOTAL/3600:.2f} hours\n")
    info += subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader; nproc; "
                           "lscpu | grep 'Model name'; free -g | head -2; df -h /tmp | tail -1; "
                           "cat /etc/os-release | head -1", shell=True, capture_output=True, text=True).stdout
    open("/kaggle/working/run_info.txt", "w").write(info)
    print(info)
    if len(done) < 50:
        print("WARNING: not all files finished - check the logs above for the missing files")